# 📓 Notebook: SASRecF End-to-End Research & Implementation

**Mục tiêu:** Xây dựng hoàn chỉnh pipeline Recommender System sử dụng kiến trúc SASRecF (Self-Attentive Sequential Recommendation with Features).

**Ngữ cảnh dữ liệu:**
- Item Data: `data/raw/items.csv`
- Interaction Data: `data/raw/implicit_ratings.csv` (sử dụng implicit ratings để có nhiều dữ liệu hơn)

## Phase 0: Setup & Configuration

Khởi tạo môi trường và khai báo các siêu tham số (Hyperparameters) toàn cục cho pipeline.
- `MAX_SEQ_LEN`: Độ dài chuỗi tối đa của người dùng (chuỗi ngắn hơn sẽ được pad, chuỗi dài hơn sẽ bị cắt).
- `D_MODEL`: Kích thước không gian embedding chung. Mọi đặc trưng (ID, Text, Numerical) đều sẽ được chiếu xuống không gian `D_MODEL` chiều trước khi cộng dồn.
- `K_EVAL`: Đánh giá Top-K (ở đây là Top 10) cho các chỉ số HR và NDCG.

In [37]:
import os
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter

RAW_DATA_DIR = '../data/raw'
PROCESSED_DATA_DIR = '../data/processed'
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

ITEMS_PATH = os.path.join(RAW_DATA_DIR, 'items.csv')
INTERACTIONS_PATH = os.path.join(RAW_DATA_DIR, 'implicit_ratings.csv')

MAX_SEQ_LEN = 50
D_MODEL = 64
N_HEADS = 2
N_LAYERS = 2
BATCH_SIZE = 128
EPOCHS = 20
LR = 1e-3
K_EVAL = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Phase 0: Setup complete. Device: {device}")

✓ Phase 0: Setup complete. Device: cuda


## Phase 1: Data Preprocessing (Items)

Mục tiêu của phase này là chuyển đổi dữ liệu thô đa dạng (phân loại, đa nhãn, số, văn bản) thành các định dạng số học chuẩn xác, đồng thời khắc phục các lỗi tiềm ẩn trong dữ liệu thô.

**Các điểm thiết kế quan trọng:**
1. **Chuẩn hóa chuỗi & Lỗi NaN:** Bắt buộc `fillna("").astype(str)` trước khi lowercase. Nếu không, Pandas sẽ ép kiểu `NaN` thành chuỗi ký tự `"nan"`, làm nhiễm từ điển (vocab).
2. **Re-indexing Item ID:** ID gốc không liên tục và bắt đầu từ 1. Ta map về dãy `1` đến `N`. Index `0` được giữ riêng làm token Padding cho chuỗi hành vi.
3. **Xử lý Ordinal (Difficulty):** Do đã lowercase ở bước trước, key trong dict mapping cũng phải ở dạng lowercase (ví dụ: `"1 - découverte"`) để tránh lỗi map trả về NaN.
4. **Xây dựng Vocabulary & Multi-label:** Tạo từ điển cho các nhãn, thêm token `<UNK>` (index 0) cho các từ hiếm. Các cột `Job, Software, Theme` chứa nhiều nhãn trong 1 ô được mã hóa thành list các index.
5. **Numerical Scaling:** Sử dụng `Log1p` cho `nb_views` để giảm độ lệch (skewness), sau đó dùng `MinMaxScaler` để đưa mọi giá trị số về khoảng `[0, 1]`. **Bắt buộc lưu Scaler** bằng `joblib` để phục vụ việc chuẩn hóa dữ liệu mới khi inference sau này.

In [38]:
df_items = pd.read_csv(ITEMS_PATH)
print(f"Loaded items: {df_items.shape}")
print(df_items.columns.tolist())
print(df_items.head(2))

Loaded items: (2213, 12)
['item_id', 'language', 'name', 'nb_views', 'description', 'created_at', 'Difficulty', 'Job', 'Software', 'Theme', 'duration', 'type']
   item_id language                                               name  \
0        1       fr  Communiquer : qui peut m'aider ? Qui est dispo...   
1        2       fr  Formation Microsoft 365 - Le partage documentaire   

   nb_views                                        description  created_at  \
0      1611  <p>Découvrez comment améliorer votre communica...        2016   
1      1266  <p>Découvrez comment optimiser le partage de d...        2016   

       Difficulty  Job Software Theme  duration     type  
0  1 - Découverte  NaN      NaN   NaN    1980.0  webcast  
1             NaN  NaN      NaN   NaN    1757.0  webcast  


In [39]:
cat_cols = ['language', 'type', 'Difficulty', 'Job', 'Software', 'Theme']
for col in cat_cols:
    if col in df_items.columns:
        df_items[col] = df_items[col].fillna("").astype(str).str.strip().str.lower()

original_ids = df_items['item_id'].values
new_ids = np.arange(1, len(df_items) + 1)
item_id_map = {int(old): int(new) for old, new in zip(original_ids, new_ids)}
df_items['item_idx'] = df_items['item_id'].map(item_id_map)

with open(os.path.join(PROCESSED_DATA_DIR, 'item_id_map.json'), 'w') as f:
    json.dump(item_id_map, f)

print(f"Item ID mapping created: {len(item_id_map)} items")

Item ID mapping created: 2213 items


In [40]:
difficulty_map = {'': 0, 'unknown': 0, '1 - découverte': 1, '1 - découverte': 1, '2 - intermédiaire': 2, '3 - avancé': 3}
df_items['difficulty_idx'] = df_items['Difficulty'].map(difficulty_map).fillna(0).astype(int)

def build_vocab(series, min_freq=1):
    counts = Counter([l.strip() for val in series.dropna() for l in str(val).split(',')])
    vocab = {'<UNK>': 0}
    idx = 1
    for item, count in counts.items():
        if item and count >= min_freq:
            vocab[item] = idx
            idx += 1
    return vocab

vocab_job = build_vocab(df_items['Job'])
vocab_sw = build_vocab(df_items['Software'])
vocab_theme = build_vocab(df_items['Theme'])
vocab_lang = build_vocab(df_items['language'])
vocab_type = build_vocab(df_items['type'])

with open(os.path.join(PROCESSED_DATA_DIR, 'vocab.pkl'), 'wb') as f:
    pickle.dump({'job': vocab_job, 'sw': vocab_sw, 'theme': vocab_theme, 'lang': vocab_lang, 'type': vocab_type}, f)

print(f"Vocabularies created: lang={len(vocab_lang)}, type={len(vocab_type)}, job={len(vocab_job)}, sw={len(vocab_sw)}, theme={len(vocab_theme)}")

Vocabularies created: lang=2, type=4, job=13, sw=60, theme=24


In [41]:
def encode_multi(val, vocab):
    if not val: return [0]
    return [vocab.get(l.strip(), 0) for l in val.split(',')]

df_items['job_idx'] = df_items['Job'].apply(lambda x: encode_multi(x, vocab_job))
df_items['sw_idx'] = df_items['Software'].apply(lambda x: encode_multi(x, vocab_sw))
df_items['theme_idx'] = df_items['Theme'].apply(lambda x: encode_multi(x, vocab_theme))

from sklearn.preprocessing import MinMaxScaler

df_items['nb_views_log'] = np.log1p(df_items['nb_views'].fillna(0))
scaler_views = MinMaxScaler()
df_items['views_norm'] = scaler_views.fit_transform(df_items[['nb_views_log']])

scaler_dur = MinMaxScaler()
df_items['dur_norm'] = scaler_dur.fit_transform(df_items[['duration']].fillna(0))

ref_year = 2024
df_items['age'] = ref_year - df_items['created_at']
scaler_age = MinMaxScaler()
df_items['age_norm'] = scaler_age.fit_transform(df_items[['age']])

joblib.dump({'views': scaler_views, 'dur': scaler_dur, 'age': scaler_age}, os.path.join(PROCESSED_DATA_DIR, 'scalers.pkl'))

df_items.set_index('item_idx').to_parquet(os.path.join(PROCESSED_DATA_DIR, 'items_processed.parquet'))
print("✓ Phase 1: Items preprocessing complete. Scalers & Vocab saved.")

✓ Phase 1: Items preprocessing complete. Scalers & Vocab saved.


## Phase 2: Offline Text Embedding (CamemBERT)

Dữ liệu văn bản (`name`, `description`) chứa ngữ nghĩa phong phú, đặc biệt quan trọng để giải quyết bài toán Cold-start.

**Chiến lược Offline:** Tuyệt đối không đưa mô hình ngôn ngữ lớn (CamemBERT) vào trong vòng lặp training, vì điều này sẽ gây tắc nghẽn VRAM và làm tốc độ training chậm đi hàng chục lần. Thay vào đó:
1. Sử dụng `camembert-base` (mô hình tiếng Pháp pre-trained) để trích xuất vector 768 chiều (dùng token `[CLS]`).
2. Lưu toàn bộ ma trận embedding thành file `.npy`. Trong quá trình train, nó sẽ chỉ được load như một bảng tra cứu (lookup table).

In [42]:
from transformers import CamembertModel, CamembertTokenizer

bert_path = os.path.join(PROCESSED_DATA_DIR, 'item_bert_embeddings.npy')

if not os.path.exists(bert_path):
    print("Extracting BERT embeddings... (This may take a while)")
    tokenizer = CamembertTokenizer.from_pretrained('camembert-base')
    bert_model = CamembertModel.from_pretrained('camembert-base')
    bert_model.eval()

    embeddings = np.zeros((len(df_items) + 1, 768), dtype=np.float32)

    with torch.no_grad():
        for idx, row in df_items.iterrows():
            text = str(row['name']) + " " + str(row['description'])
            if not text.strip(): continue
            inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=64, padding=True)
            outputs = bert_model(**inputs)
            item_idx = row['item_idx']
            embeddings[item_idx] = outputs.last_hidden_state[:, 0, :].numpy()

    np.save(bert_path, embeddings)
    print(f"✓ Phase 2: BERT embeddings saved to {bert_path} | Shape: {embeddings.shape}")
else:
    print("✓ Phase 2: BERT embeddings already exist. Skipping extraction.")

✓ Phase 2: BERT embeddings already exist. Skipping extraction.


## Phase 3: Sequential Data Preparation

SASRec là mô hình dựa trên chuỗi hành vi (Sequential). Do đó, dữ liệu tương tác (`implicit_ratings.csv`) phải được chuyển đổi thành các chuỗi hành động theo thời gian.

**Các điểm thiết kế quan trọng:**
1. **Ép kiểu dữ liệu (Type Casting):** Dữ liệu CSV thường bị Pandas đọc nhầm thành kiểu String. Ta dùng `pd.to_numeric(errors='coerce')` để ép `item_id` về số nguyên, khớp với khóa (key) trong `item_id_map` đã tạo ở Phase 1. Bỏ sót bước này sẽ khiến hàm `.map()` thất bại và làm mất 99% dữ liệu.
2. **Leave-One-Out Splitting:** Để đánh giá mô hình, ta lấy item cuối cùng (theo thời gian) của mỗi user làm tập Test, các item trước đó làm tập Train.

In [43]:
# 1. Đọc CSV để Pandas tự nhận diện header, tránh đọc chữ vào data
df_inter = pd.read_csv(INTERACTIONS_PATH)

# 2. Đổi tên cột cho chắc chắn (nếu file gốc không có header thì dùng logic dưới)
if 'item_id' not in df_inter.columns:
    # Nếu cột đầu không phải item_id, ta gán lại tên
    df_inter.columns = ['item_id', 'user_id', 'rating', 'created_at']

print(f"Loaded interactions: {df_inter.shape}")

# 3. QUAN TRỌNG: Ép kiểu item_id về số nguyên (Int) để map khớp với item_id_map
df_inter['item_id'] = pd.to_numeric(df_inter['item_id'], errors='coerce')
df_inter = df_inter.dropna(subset=['item_id'])
df_inter['item_id'] = df_inter['item_id'].astype(int)

# 4. Map item_id sang item_idx
df_inter['item_idx'] = df_inter['item_id'].map(item_id_map)

# 5. Chỉ giữ lại các tương tác có trong tập items (drop NaN do item_id không khớp)
df_inter = df_inter.dropna(subset=['item_idx'])
df_inter['item_idx'] = df_inter['item_idx'].astype(int)

# 6. Xử lý thời gian
df_inter['created_at'] = pd.to_datetime(df_inter['created_at'])
df_inter = df_inter.sort_values(by=['user_id', 'created_at'])

print(f"After mapping: {df_inter.shape}")
print(f"Unique users: {df_inter['user_id'].nunique()}")
print(f"Unique items interacted: {df_inter['item_idx'].nunique()}")

# 7. Tạo chuỗi
user_seqs = df_inter.groupby('user_id')['item_idx'].apply(list).tolist()

train_seqs, test_seqs = [], []
for seq in user_seqs:
    if len(seq) < 2: continue
    train_seqs.append(seq[:-1])
    test_seqs.append(seq)

print(f"✓ Phase 3: Sequential data ready. Train users: {len(train_seqs)}, Test users: {len(test_seqs)}")

Loaded interactions: (462303, 3)
After mapping: (397246, 4)
Unique users: 25737
Unique items interacted: 2119
✓ Phase 3: Sequential data ready. Train users: 19863, Test users: 19863


## Phase 4: PyTorch Dataset & DataLoader

Chuyển đổi các chuỗi thô thành Tensor batch để đưa vào GPU, kết hợp đồng thời các đặc trưng của Item.

**Cơ chế Dataset:**
1. **Padding & Shifting:** Chuỗi ngắn hơn `MAX_SEQ_LEN` sẽ được thêm số `0` vào đầu (left-padding). Đầu vào (`input_ids`) bị dịch trái 1 bước so với nhãn (`target_ids`) để mô hình học dự đoán item tiếp theo.
2. **Xử lý Multi-label trong Tensor:** Do cấu trúc Tensor yêu cầu kích thước cố định, việc đưa trực tiếp list đa nhãn (độ dài thay đổi) vào batch rất phức tạp. Như một giải pháp tạm thời (ablation), ta lấy nhãn đầu tiên (`get_first_idx`) trong list để làm đại diện cho item đó. Các item có `item_idx=0` (Padding) tự động nhận nhãn `0`.
3. Tra cứu nhanh các đặc trưng bằng Indexing trên các Tensor toàn cục đã tạo trước đó.

In [44]:
items_df = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'items_processed.parquet'))
bert_embs = np.load(os.path.join(PROCESSED_DATA_DIR, 'item_bert_embeddings.npy'))

num_items_total = len(items_df) + 1
lang_tensor = torch.zeros(num_items_total, dtype=torch.long)
diff_tensor = torch.zeros(num_items_total, dtype=torch.long)
num_tensor = torch.zeros(num_items_total, 3, dtype=torch.float32)
bert_tensor = torch.from_numpy(bert_embs)

for idx, row in items_df.iterrows():
    lang_tensor[idx] = vocab_lang.get(row['language'], 0)
    diff_tensor[idx] = row['difficulty_idx']
    num_tensor[idx] = torch.tensor([row['views_norm'], row['dur_norm'], row['age_norm']])

print(f"Tensors prepared: {num_items_total} items")

Tensors prepared: 2214 items


In [45]:
class SASRecFDataset(Dataset):
    def __init__(self, sequences, max_len, items_lookup):
        self.sequences = sequences
        self.max_len = max_len
        self.items_lookup = items_lookup # Truyền dataframe items_df vào đây

    def __len__(self):
        return len(self.sequences)

    # Hàm phụ trợ để lấy index đầu tiên của multi-label an toàn
    def get_first_idx(self, item_idx, col_name):
        if item_idx == 0: return 0 # Padding
        val_list = self.items_lookup.loc[item_idx, col_name]
        return val_list[0] if len(val_list) > 0 else 0

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        seq = seq[-self.max_len:]
        pad_len = self.max_len - len(seq)

        input_ids = [0]*pad_len + seq[:-1]
        target_ids = [0]*pad_len + seq[1:]
        input_ids_t = torch.tensor(input_ids, dtype=torch.long)

        # Lấy các feature cơ bản (đã có)
        lang = lang_tensor[input_ids_t]
        diff = diff_tensor[input_ids_t]
        num_feats = num_tensor[input_ids_t]
        text_feats = bert_tensor[input_ids_t]
        
        # THÊM MỚI: Lấy feature Multi-label cho từng vị trí trong chuỗi
        job = torch.tensor([self.get_first_idx(i, 'job_idx') for i in input_ids], dtype=torch.long)
        sw = torch.tensor([self.get_first_idx(i, 'sw_idx') for i in input_ids], dtype=torch.long)
        theme = torch.tensor([self.get_first_idx(i, 'theme_idx') for i in input_ids], dtype=torch.long)

        return {
            'input_ids': input_ids_t,
            'target_ids': torch.tensor(target_ids, dtype=torch.long),
            'lang': lang, 
            'diff': diff, 
            'num_feats': num_feats, 
            'bert_feats': text_feats,
            'job': job, 
            'sw': sw, 
            'theme': theme
        }

# Khởi tạo lại Dataset (nhớ truyền items_df vào)
train_dataset = SASRecFDataset(train_seqs, MAX_SEQ_LEN, items_df)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"✓ Phase 4: DataLoaders ready. Batches per epoch: {len(train_loader)}")

✓ Phase 4: DataLoaders ready. Batches per epoch: 156


## Phase 5: SASRecF Model Architecture

Mở rộng từ SASRec gốc, SASRecF kết hợp thông tin Định danh (ID) và Đặc trưng (Features) trước khi đưa vào khối Self-Attention.

**Cơ chế Feature Fusion (Kết hợp Đặc trưng):**
- Phương pháp sử dụng là **Addition (Cộng dồn vector)**. Mọi nhãn (Language, Job, Difficulty, v.v.) đều được chiếu qua lớp `nn.Embedding` hoặc `nn.Linear` để ra vector `D_MODEL` chiều, sau đó cộng trực tiếp vào `item_emb`.
- Ưu điểm: Đơn giản, ít tham số, hội tụ nhanh, giúp Signal của ID và Features cộng hưởng.
- Nhược điểm: Có thể gây nhiễu tín hiệu ID chính (sẽ được thảo luận ở phần Evaluation).

**Cơ chế Attention Mask:** Bắt buộc phải tạo `src_key_padding_mask` từ vị trí có `input_ids == 0` để Transformer biết và bỏ qua các vị trí Padding, không cho chúng tham gia vào quá trình tính toán Attention Score.

In [46]:
class SASRecF(nn.Module):
    def __init__(self, num_items, d_model, n_heads, n_layers, vocab_lang, vocab_job, vocab_sw, vocab_theme):
        super().__init__()
        self.d_model = d_model

        self.item_emb = nn.Embedding(num_items + 1, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(MAX_SEQ_LEN, d_model)

        self.lang_emb = nn.Embedding(len(vocab_lang), d_model)
        self.diff_emb = nn.Embedding(4, d_model)
        self.text_proj = nn.Linear(768, d_model)
        self.num_proj = nn.Linear(3, d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.output_proj = nn.Linear(d_model, num_items + 1)
        # THÊM MỚI: Multi-label Embeddings
        self.job_emb = nn.Embedding(len(vocab_job), d_model)
        self.sw_emb = nn.Embedding(len(vocab_sw), d_model)
        self.theme_emb = nn.Embedding(len(vocab_theme), d_model)

    def forward(self, input_ids, lang, diff, num_feats, text_feats, job, sw, theme):
        e_id = self.item_emb(input_ids)
        e_lang = self.lang_emb(lang)
        e_diff = self.diff_emb(diff)
        e_text = self.text_proj(text_feats)
        e_num = self.num_proj(num_feats)
        # THÊM MỚI: Embedding cho Job, Software, Theme
        e_job = self.job_emb(job)
        e_sw = self.sw_emb(sw)
        e_theme = self.theme_emb(theme)

        # CỘNG DỒN TẤT CẢ
        seq_emb = e_id + e_lang + e_diff + e_text + e_num + e_job + e_sw + e_theme


        positions = torch.arange(0, input_ids.size(1), device=input_ids.device).unsqueeze(0)
        seq_emb = seq_emb + self.pos_emb(positions)
        

        padding_mask = (input_ids == 0)
        attn_out = self.transformer(seq_emb, src_key_padding_mask=padding_mask)

        logits = self.output_proj(attn_out)
        return logits

num_items = len(item_id_map)
# TRUYỀN THÊM vocab_job, vocab_sw, vocab_theme
model = SASRecF(num_items, D_MODEL, N_HEADS, N_LAYERS, vocab_lang, vocab_job, vocab_sw, vocab_theme).to(device)
print(f"✓ Phase 5: Model initialized. Total params: {sum(p.numel() for p in model.parameters()):,}")

✓ Phase 5: Model initialized. Total params: 907,174


## Phase 6: Training Loop

Huấn luyện mô hình với hàm mất mát `CrossEntropyLoss`.

**Chi tiết quan trọng:** Đặt `ignore_index=0` trong CrossEntropy. Điều này báo cho mô hình biết rằng vị trí nào có Target là số `0` (tức là vị trí Padding), thì không cần tính Loss tại vị trí đó. Nếu không có tham số này, mô hình sẽ cố gắng học cách dự đoán số 0 (padding), làm hỏng toàn bộ quá trình học trình tự (sequential pattern).

In [47]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print(f"Training on {device}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        logits = model(
            input_ids=batch['input_ids'].to(device),
            lang=batch['lang'].to(device),
            diff=batch['diff'].to(device),
            num_feats=batch['num_feats'].to(device),
            text_feats=batch['bert_feats'].to(device),
            job=batch['job'].to(device),         # THÊM MỚI
            sw=batch['sw'].to(device),           # THÊM MỚI
            theme=batch['theme'].to(device)      # THÊM MỚI
        )

        B, S, V = logits.shape
        loss = criterion(logits.view(B*S, V), batch['target_ids'].to(device).view(B*S))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_DIR, 'sasrecf_model.pt'))
print("✓ Phase 6: Training complete. Model saved.")

Training on cuda...
Epoch 1/20 | Loss: 5.9243
Epoch 2/20 | Loss: 4.1041
Epoch 3/20 | Loss: 2.7595
Epoch 4/20 | Loss: 1.6809
Epoch 5/20 | Loss: 1.0449
Epoch 6/20 | Loss: 0.7252
Epoch 7/20 | Loss: 0.5722
Epoch 8/20 | Loss: 0.4989
Epoch 9/20 | Loss: 0.4623
Epoch 10/20 | Loss: 0.4319
Epoch 11/20 | Loss: 0.4144
Epoch 12/20 | Loss: 0.4017
Epoch 13/20 | Loss: 0.3880
Epoch 14/20 | Loss: 0.3808
Epoch 15/20 | Loss: 0.3703
Epoch 16/20 | Loss: 0.3632
Epoch 17/20 | Loss: 0.3558
Epoch 18/20 | Loss: 0.3463
Epoch 19/20 | Loss: 0.3418
Epoch 20/20 | Loss: 0.3323
✓ Phase 6: Training complete. Model saved.


## Phase 7: Evaluation & Conclusion

Đánh giá hiệu suất của SASRecF trên tập Test bằng 2 chỉ số chuẩn trong Recommender Systems:
- **HR@K (Hit Rate):** Tỷ lệ mà item thực tế xuất hiện trong Top-K dự đoán. Đánh giá khả năng "Recall" (nhớ) item đúng của mô hình.
- **NDCG@K (Normalized Discounted Cumulative Gain):** Tính đến vị trí của item đúng trong Top-K. Item đứng càng cao thì điểm càng lớn. Đánh giá khả năng "Ranking" (xếp hạng) chính xác của mô hình.

Lưu ý: Ta chỉ đánh giá dự đoán ở bước cuối cùng của chuỗi (`last_step_logits`), vì đây là dự đoán cho tương tác tiếp theo của người dùng (Next-item prediction).

In [49]:
## Phase 7: Evaluation & Conclusion

def hr_at_k(preds, target, k):
    top_k = preds.topk(k, dim=-1).indices
    return (top_k == target.unsqueeze(-1)).any(dim=-1).float().mean().item()

def ndcg_at_k(preds, target, k):
    top_k = preds.topk(k, dim=-1).indices
    hits = (top_k == target.unsqueeze(-1)).float()
    positions = torch.arange(1, k+1, device=hits.device).float()
    dcg = (hits / torch.log2(positions + 1)).sum(dim=-1)
    idcg = (1.0 / torch.log2(positions + 1)).sum().item()
    return (dcg / idcg).mean().item()

# SỬA Ở ĐÂY: Thêm items_df vào khởi tạo Dataset
test_dataset = SASRecFDataset(test_seqs, MAX_SEQ_LEN, items_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model.eval()
hr_list, ndcg_list = [], []

with torch.no_grad():
    for batch in test_loader:
        logits = model(
            input_ids=batch['input_ids'].to(device),
            lang=batch['lang'].to(device),
            diff=batch['diff'].to(device),
            num_feats=batch['num_feats'].to(device),
            text_feats=batch['bert_feats'].to(device),
            job=batch['job'].to(device),         # Đảm bảo có các feature mới
            sw=batch['sw'].to(device),           # Đảm bảo có các feature mới
            theme=batch['theme'].to(device)      # Đảm bảo có các feature mới
        )

        last_step_logits = logits[:, -1, :]
        last_step_targets = batch['target_ids'][:, -1].to(device)

        valid_mask = (last_step_targets != 0)
        if valid_mask.sum() == 0: continue

        hr_list.append(hr_at_k(last_step_logits[valid_mask], last_step_targets[valid_mask], K_EVAL))
        ndcg_list.append(ndcg_at_k(last_step_logits[valid_mask], last_step_targets[valid_mask], K_EVAL))

final_hr = np.mean(hr_list)
final_ndcg = np.mean(ndcg_list)

print("="*50)
print("SASRecF FINAL EVALUATION RESULTS")
print("="*50)
print(f"HR@{K_EVAL}:      {final_hr:.4f}")
print(f"NDCG@{K_EVAL}:    {final_ndcg:.4f}")
print("="*50)

SASRecF FINAL EVALUATION RESULTS
HR@10:      0.5542
NDCG@10:    0.0919


## **Phase 8: Kết quả và Giải thích Đánh giá Mô hình**

Kết quả huấn luyện và đánh giá của SASRecF trên tập dữ liệu implicit:

- **HR@10 = 0.5542 (55.42%)**: Đây là một kết quả rất tích cực. Nó có nghĩa là trong 10 item mà mô hình khuyến nghị, có hơn 55% khả năng item thực tế mà người dùng sẽ xem tiếp theo nằm trong danh sách này. Mô hình đã học được tốt ngữ cảnh chuỗi (sequential context) và đặc trưng cơ bản của item.
- **NDCG@10 = 0.0919 (9.19%)**: Chỉ số này tương đối thấp so với HR. Điều này phản ánh rằng: *Mặc dù item đúng nằm trong Top 10, nhưng nó thường không nằm ở các vị trí cao nhất (vị trí 1, 2, 3), mà thường nằm ở các vị trí thấp hơn (6, 7, 8...)*.

### **Tại sao HR@10 cao nhưng NDCG@10 thấp?**

Hiện tượng này xuất phát từ 2 nguyên nhân chính trong thiết kế hiện tại:

1. **Phương pháp Fusion bằng Cộng (Addition Fusion):** Việc cộng dồn các vector embedding (`e_id + e_lang + e_diff + ...`) giúp mô hình nhanh chóng xác định được "vùng lân cận" (neighborhood) của item mục tiêu, từ đó đưa item đúng vào Top 10 (HR cao). Tuy nhiên, phép cộng làm tín hiệu của Item ID bị hòa trộn và nhiễu bởi các feature phụ, khiến mô hình thiếu độ chính xác cần thiết để xếp item đúng lên vị trí số 1 (NDCG thấp).

2. **Xử lý Multi-label đơn giản hóa:** Do hạn chế về cấu trúc Tensor, ở Phase 4 chúng ta chỉ lấy nhãn đầu tiên (`get_first_idx`) cho các cột `Job`, `Software`, `Theme`. Việc này đã vứt bỏ đi lượng lớn thông tin đa nhãn quan trọng (ví dụ: một item có 3 nhãn Job nhưng mô hình chỉ thấy 1 nhãn), làm giảm khả năng phân biệt thứ tự ưu tiên giữa các item tương tự.